In [17]:
# !pip install langchain langgraph langchain-openai python-dotenv

import os
import uuid
from datetime import datetime
from dotenv import load_dotenv
load_dotenv()


from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.store.memory import InMemoryStore
from langgraph.config import get_store



openrouter_base_url = os.getenv("OPENROUTER_BASE_URL")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if not openrouter_base_url or not openrouter_api_key:
    raise ValueError("OPENROUTER_BASE_URL or OPENROUTER_API_KEY is not set")

## Інструменти:
   * Додавання завдань
   * Перегляд всіх завдань
   * Видалення завдання за ID

In [18]:
@tool
def add_task(title: str) -> str:
    """Додає нове завдання до списку з унікальним ID.

    Args:
        title: Назва або зміст завдання.
    """
    store = get_store()
    task_id = str(uuid.uuid4())
    store.put(("tasks",), task_id, {"title": title, "id": task_id})
    return f'Завдання "{title}" було успішно додано (ID: {task_id[:8]}, повний ID: {task_id}).'


@tool
def list_tasks() -> str:
    """Показує всі збережені завдання з номерами та ID."""
    store = get_store()
    items = store.search(("tasks",))
    if not items:
        return "Список завдань порожній."

    result = ["Ось всі ваші завдання:"]
    for idx, item in enumerate(items, 1):
        task_data = item.value
        title = task_data.get("title", "")
        short_id = item.key[:8]
        result.append(f"{idx}. {title} [{short_id}] (ID: {item.key})")
    return "\n".join(result)


@tool
def delete_task(task_id: str) -> str:
    """Видаляє завдання за його ID.

    Args:
        task_id: Унікальний ідентифікатор завдання (ID).
    """
    store = get_store()
    item = store.get(("tasks",), task_id)
    if not item:
        # Якщо передано короткий ID або частину ключа, виконаємо пошук серед завдань
        items = store.search(("tasks",))
        for it in items:
            if it.key.startswith(task_id):
                item = it
                task_id = it.key
                break

    if not item:
        return f"Завдання з ID '{task_id}' не знайдено."

    title = item.value.get("title", "")
    store.delete(("tasks",), task_id)

    return f'Завдання "{title}" було видалено.'

## Агент з інструментами та store

In [19]:
store = InMemoryStore()

llm = ChatOpenAI(
    model="openai/gpt-4o-mini" if openrouter_base_url and "openrouter" in openrouter_base_url else "gpt-4o-mini",
    api_key=openrouter_api_key,
    base_url=openrouter_base_url,
    temperature=0
)

system_prompt = (
    "Ти — корисний помічник (TODO-агент) для керування списком справ. "
    "Ти вмієш додавати, переглядати та видаляти завдання за допомогою наданих інструментів. "
    "Усі завдання зберігаються у сховищі. "
    "Завжди відповідай українською мовою. "
    "Коли користувач просить видалити завдання за назвою або описом, спочатку переглянь список завдань через list_tasks, щоб знайти відповідний ID, а потім виклич delete_task."
)

agent = create_agent(
    model=llm,
    tools=[add_task, list_tasks, delete_task],
    store=store,
    system_prompt=system_prompt
)

## Тестування агента

In [20]:
tests = [
    "Додай: купити хліб",
    "Додай: подзвонити лікарю",
    "Покажи всі завдання",
    "Видали завдання про хліб",
    "Видали завдання про дитячий садочок",
    "Що залишилось?"
]

messages = []
for i, test in enumerate(tests, 1):
    print(f"\n{'='*50}")
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S} TEST {i}: {test}")
    print('='*50)
    messages.append({"role": "user", "content": test})
    result = agent.invoke({"messages": messages})
    messages = result["messages"]
    last_msg = messages[-1].content
    print(last_msg)


2026-11-03 18:10:33 TEST 1: Додай: купити хліб
Завдання "купити хліб" було успішно додано до списку. Його ID: ffdaa78c.

2026-11-03 18:10:35 TEST 2: Додай: подзвонити лікарю
Завдання "подзвонити лікарю" було успішно додано до списку. Його ID: d7756c58.

2026-11-03 18:10:38 TEST 3: Покажи всі завдання
Ось всі ваші завдання:

1. купити хліб [ffdaa78c] (ID: ffdaa78c-6b1f-4936-b2de-e2568d546134)
2. подзвонити лікарю [d7756c58] (ID: d7756c58-918e-42b4-b5b1-00c6a2ba7235)

2026-11-03 18:10:41 TEST 4: Видали завдання про хліб
Завдання "купити хліб" було успішно видалено.

2026-11-03 18:10:43 TEST 5: Видали завдання про дитячий садочок
У вас немає завдання про дитячий садочок. Ось ваші поточні завдання:

1. подзвонити лікарю [d7756c58] (ID: d7756c58-918e-42b4-b5b1-00c6a2ba7235)

2026-11-03 18:10:46 TEST 6: Що залишилось?
У вас залишилось одне завдання:

1. подзвонити лікарю [d7756c58] (ID: d7756c58-918e-42b4-b5b1-00c6a2ba7235)
